In [1]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchsummary import summary
from PIL import Image

import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Subset, random_split
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader

from load_data import  CatDogDataLoadandSave, split_dataset
from vgg16_model import VGG16
from data_classes import TrainingConfig


### Load data 


In [2]:
data_path = r'/Users/gimoon/Documents/GitHub/Data'
train_data_path = os.path.join(data_path, "cat-and-dog", "training_set")
test_data_path  = os.path.join(data_path, "cat-and-dog", "test_set")

assert os.path.exists(train_data_path) == True, f"Training data path does not exist: {train_data_path}"
assert os.path.exists(test_data_path) == True, f"Test data path does not exist: {test_data_path}"

### create dataset 

In [3]:
# Load full dataset
dataset_train = CatDogDataLoadandSave(data_dir=train_data_path)


# Create a subset with reduced number of images
num_train_samples = 400  # Reduce to 1000 images (adjust as needed)
indices = list(range(min(num_train_samples, len(dataset_train))))
dataset_train_subset = Subset(dataset_train, indices)

print(f"Original dataset size: {len(dataset_train)}")
print(f"Subset dataset size: {len(dataset_train_subset)}")


Original dataset size: 8005
Subset dataset size: 400


### Split dataset into training and validation sets, then create DataLoaders

In [4]:

train_dataset, val_dataset = split_dataset(dataset_train_subset, train_ratio=0.8)

# Configuration for training
config = TrainingConfig(
    batch_size=32,
    num_epochs=1,
    num_train_samples=len(train_dataset),  # Set from your dataset
    lr_scheduler_epoch=5
)
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training set size: 320
Validation set size: 80

Train batches: 10
Validation batches: 3


### Build VGG16 architecture

In [5]:
### Configure loss function, optimizer, and learning rate scheduler
if torch.backends.mps.is_available():    
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model = VGG16(3, 2).to(device)
device

device(type='mps')

In [6]:
from torchsummary import summary

if device.type == 'cpu':
    summary(model, (3, 224, 224))


### set loss criterion and optimizer 

In [7]:
lr_scheduler_step_size = config.lr_scheduler_step_size()
print(lr_scheduler_step_size)

total_steps = config.total_steps()
print(total_steps)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
scheduler = StepLR(optimizer, step_size=config.lr_scheduler_step_size, gamma=0.5)

50
10


In [8]:

for epoch in range(config.num_epochs):
    ProgressBar = tqdm(enumerate(train_loader), total=len(train_loader))

    for batch_idx, (inputs, labels) in ProgressBar:
        
        model.train()
        # Ensure labels are torch.long before moving to device for CrossEntropyLoss
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # validate 
        model.eval()
        val_loss = 0.
        num_correct = 0
        num_samples = 0
        
        with torch.no_grad():
            for inputs_val, labels_val in val_loader:
                inputs_val, labels_val = inputs_val.to(device), labels_val.to(device)
                outputs_val = model(inputs_val)
                loss_val = criterion(outputs_val, labels_val)
                val_loss += loss_val.item()
                _, predictions = outputs_val.max(1)
                num_correct += (predictions == labels_val).sum()
                num_samples += predictions.size(0)

            avg_val_loss = val_loss / len(val_loader)
            avg_val_acc = num_correct / num_samples if num_samples > 0 else 0
            

        #Update Progress bar
        ProgressBar.set_description(f'Epoch [{epoch+1}]')
        ProgressBar.set_postfix(TrainLoss=loss.item(), ValLoss=avg_val_loss, ValAcc=avg_val_acc)


Epoch [1]: 100%|██████████| 10/10 [00:26<00:00,  2.60s/it, TrainLoss=0, ValAcc=tensor(1., device='mps:0'), ValLoss=0]          


In [10]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)
            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)

    model.train()
    return num_correct/num_samples


In [11]:
test_loader = DataLoader(CatDogDataLoadandSave(test_data_path))

check_accuracy(val_loader, model)

tensor(1., device='mps:0')